#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim,col
from pyspark.sql.window import Window

In [0]:
RENAME_MAP = {
    "CID": "customer_key",
    "CNTRY": "country"
}

#Reading From Bronze

In [0]:
df = spark.table("workspace.bronze.erp_loc_a101")
df.display()

# Data Transformations

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Null handling and Empty values

In [0]:
def is_empty(col):
    return col.isNull() | (F.trim(col) == "")

condition_invalid = (
    is_empty(F.col("CID")) |
    is_empty(F.col("CNTRY")) 
)

df_valid = df.filter(~condition_invalid)
df_invalid = df.filter(condition_invalid)

invalid_count = df_invalid.count()

print("Total rows:", df.count())
print("Invalid rows:", df_invalid.count())
print("Valid rows:", df_valid.count())

df = df_valid

## Handle duplicate ids

In [0]:
duplicate_cid_count = (
    df.groupBy("CID")
      .count()
      .filter(F.col("count") > 1)
      .count()
)

print("Duplicate CID:", duplicate_cid_count)

duplicate_conflicts = (
    df.groupBy("CID")
      .agg(F.countDistinct("CNTRY").alias("country_count"))
      .filter(F.col("country_count") > 1)
      .count()
)

print("Duplicate CID with conflicting countries:", duplicate_conflicts)

df = df.dropDuplicates(["CID", "CNTRY"])

duplicate_after = (
    df.groupBy("CID")
      .count()
      .filter(F.col("count") > 1)
      .count()
)

print("Duplicate CID after cleaning:", duplicate_after)


##Normalization

In [0]:
df = df.withColumn(
    "CNTRY",
    F.when(F.upper(F.trim(F.col("CNTRY"))).isin("US", "USA", "UNITED STATES"), "United States")
     .when(F.upper(F.trim(F.col("CNTRY"))).isin("UNITED KINGDOM"), "United Kingdom")
     .when(F.upper(F.trim(F.col("CNTRY"))).isin("DE", "GERMANY"), "Germany")
     .when(F.upper(F.trim(F.col("CNTRY"))).isin("CANADA"), "Canada")
     .when(F.upper(F.trim(F.col("CNTRY"))).isin("AUSTRALIA"), "Australia")
     .when(F.upper(F.trim(F.col("CNTRY"))).isin("FRANCE"), "France")
     .otherwise("Unknown")
)

df.select("CNTRY").distinct().orderBy("CNTRY").display()

unknown_count = df.filter(F.col("CNTRY") == "Unknown").count()
print("Unknown country count:", unknown_count)


## Renamig the columns

In [0]:
for old_name,new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity checks before write

In [0]:
def sanity_check(df):
    row_count = df.count()

    duplicate_customer_key = (
        df.groupBy("customer_key")
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    null_critical = (
        df.filter(
            F.col("customer_key").isNull() |
            F.col("country").isNull()
        ).count()
    )

    return {
        "row_count": row_count,
        "duplicate_customer_key": duplicate_customer_key,
        "null_critical": null_critical
    }

results = sanity_check(df)
print("Before write:", results)

# Write Into Silver

In [0]:
(df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver.erp_locations"))

df_silver = spark.table("workspace.silver.erp_locations")

results = sanity_check(df_silver)
print("After write:", results)

if results["duplicate_customer_key"] > 0:
    raise Exception("Duplicate customer_key found!")

if results["null_critical"] > 0:
    raise Exception("Null critical fields found!")

